# ⚙️ Product Design Optimization — Design of Experiments (DOE)
**Goal:** Identify optimal factor settings to maximize performance while minimizing cost  
**Method:** 2^4 Full Factorial Design with ANOVA — 16 treatment combinations  
**Output:** Power BI reporting dashboard showing main effects and interaction plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from itertools import product
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print('Libraries loaded')

In [ ]:
# 2^4 Full Factorial Design
# Factors: Material (A), Temperature (B), Pressure (C), Speed (D)
# Response: Performance Score (higher is better), Unit Cost ($)

factors = {'Material': [-1, 1],      # -1=Standard, +1=Premium
           'Temperature': [-1, 1],   # -1=Low (180°C), +1=High (220°C)
           'Pressure': [-1, 1],      # -1=Low (50 bar), +1=High (80 bar)
           'Speed': [-1, 1]}         # -1=Slow (100 rpm), +1=Fast (200 rpm)

# Generate all 16 combinations
design_matrix = pd.DataFrame(list(product(*factors.values())), columns=factors.keys())

# Simulate responses with known effects + interaction
# Main effects: Material (+8), Temp (+5), Pressure (+3), Speed (-2)
# Interaction: Material*Temp (+4)
np.random.seed(42)
design_matrix['Performance'] = (72
    + 8 * design_matrix['Material']
    + 5 * design_matrix['Temperature']
    + 3 * design_matrix['Pressure']
    - 2 * design_matrix['Speed']
    + 4 * design_matrix['Material'] * design_matrix['Temperature']
    + np.random.normal(0, 1.5, 16)).round(2)

design_matrix['Unit_Cost'] = (45
    + 12 * (design_matrix['Material']+1)/2
    + 3 * (design_matrix['Temperature']+1)/2
    + 2 * (design_matrix['Pressure']+1)/2
    + np.random.normal(0, 1.2, 16)).round(2)

# Replicate 3 times for ANOVA
replicates = []
for _ in range(3):
    rep = design_matrix.copy()
    rep['Performance'] += np.random.normal(0, 1.0, 16)
    rep['Unit_Cost'] += np.random.normal(0, 0.5, 16)
    replicates.append(rep)
df_full = pd.concat(replicates, ignore_index=True)

print(f'DOE Design: 2^4 Full Factorial | {len(design_matrix)} runs | 3 replicates = {len(df_full)} total')
print(f'Performance range: {df_full["Performance"].min():.1f} – {df_full["Performance"].max():.1f}')
design_matrix.head(8)

In [ ]:
# Main Effects Plot
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Main Effects Plot — Performance Score (DOE 2^4)', fontsize=14, fontweight='bold')

factor_labels = {'Material': ['Standard (-1)', 'Premium (+1)'],
                 'Temperature': ['Low 180°C (-1)', 'High 220°C (+1)'],
                 'Pressure': ['Low 50bar (-1)', 'High 80bar (+1)'],
                 'Speed': ['Slow 100rpm (-1)', 'Fast 200rpm (+1)']}

for ax, factor in zip(axes, factors.keys()):
    means = df_full.groupby(factor)['Performance'].mean()
    sems = df_full.groupby(factor)['Performance'].sem()
    ax.bar([-1, 1], means.values, width=0.6, color=['#B5D4F4','#1D9E75'],
           edgecolor='white', yerr=sems.values, capsize=5)
    ax.set_xticks([-1, 1])
    ax.set_xticklabels(['-1', '+1'], fontsize=10)
    ax.set_title(factor, fontsize=12, fontweight='500')
    ax.set_ylabel('Avg Performance' if factor == 'Material' else '')
    effect = means[1] - means[-1]
    ax.text(0, max(means.values) + 1.5, f'Effect: {effect:+.1f}', ha='center', fontsize=10,
            color='#1D9E75' if effect > 0 else '#E24B4A', fontweight='500')
    ax.set_ylim(55, 100)

plt.tight_layout()
plt.savefig('../outputs/main_effects_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interaction Plot — Material x Temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Interaction Analysis — DOE 2^4', fontsize=14, fontweight='bold')

# Material x Temperature interaction
for mat_level, color, label in [(-1, '#378ADD', 'Standard Material'), (1, '#1D9E75', 'Premium Material')]:
    subset = df_full[df_full['Material'] == mat_level]
    means = subset.groupby('Temperature')['Performance'].mean()
    axes[0].plot([-1, 1], means.values, 'o-', color=color, linewidth=2.5, markersize=8, label=label)

axes[0].set_xticks([-1, 1])
axes[0].set_xticklabels(['Low Temp (180°C)', 'High Temp (220°C)'])
axes[0].set_ylabel('Avg Performance')
axes[0].set_title('Material × Temperature Interaction\n(Lines NOT parallel = significant interaction)')
axes[0].legend()

# ANOVA summary
anova_results = {}
grand_mean = df_full['Performance'].mean()
for factor in factors.keys():
    groups = [df_full[df_full[factor]==lvl]['Performance'].values for lvl in [-1, 1]]
    f, p = stats.f_oneway(*groups)
    anova_results[factor] = {'F-stat': round(f, 2), 'p-value': round(p, 4), 'Significant': 'Yes' if p < 0.05 else 'No'}

anova_df = pd.DataFrame(anova_results).T
axes[1].axis('off')
table = axes[1].table(cellText=anova_df.reset_index().values,
                       colLabels=['Factor','F-statistic','p-value','Significant?'],
                       cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)
# Color significant rows
for (row, col), cell in table.get_celld().items():
    if row > 0 and col == 3:
        cell.set_facecolor('#EAF3DE' if cell.get_text().get_text() == 'Yes' else '#FCEBEB')
axes[1].set_title('ANOVA Results', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../outputs/interaction_anova.png', dpi=150, bbox_inches='tight')
plt.show()

# Optimal settings
print('\n=== OPTIMAL FACTOR SETTINGS ===')
print('Material: Premium (+1) | Temperature: High 220°C (+1) | Pressure: High 80bar (+1) | Speed: Slow 100rpm (-1)')
optimal = df_full[(df_full['Material']==1)&(df_full['Temperature']==1)&(df_full['Pressure']==1)&(df_full['Speed']==-1)]
print(f'Predicted Performance: {optimal["Performance"].mean():.1f} | Unit Cost: ${optimal["Unit_Cost"].mean():.2f}')

# Export for Power BI
df_full.to_csv('../outputs/doe_results.csv', index=False)
print('Exported doe_results.csv — ready for Power BI reporting dashboard')